In [1]:
%pip install langchain langchain-community faiss-cpu sentence-transformers
%pip install langchain-huggingface langchain-google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/724.4 kB ? eta -:--:--
   ---------------------------------------- 724.4/724.4 kB 16.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 43.8 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ----------------------------------------  0/11 [filetype]
   ------- --------------------------------  2/11 [pycparser]
   ---------- -----------------------------  3/11 [pyasn1]
   ---------- -----------------------------  3/11 [pyasn1]
   -------------- -------------------------  4/11 [rsa]
   -------------- -------------------------  4/11 [rsa]
   -------------- -------------------------  4/11 [rsa]
   -------------- -------------------------  4/11 [rsa]
   ------------------ ---------------------  5/11 [pyasn1-modules]
   ------------------ ---------------------  5/11 [pyasn1-m


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document


In [3]:
load_dotenv(Path("env/.env"))
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [4]:
# ── CONFIGURACIÓN GLOBAL ──────────────────────────────────────────────────
FAISS_DIR   = "faiss_index"    # Carpeta donde está el índice FAISS
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TOP_K       = 4             # Número de chunks a recuperar por consulta
GEMINI_MODEL = "models/gemini-2.5-flash-lite"  # Modelo Gemini a usar

In [5]:
# ── CARGAR ÍNDICE FAISS ────────────────────────────────────────────────────
def cargar_vectorstore(faiss_dir: str) -> FAISS:
    """
    Carga el índice FAISS ya creado en el paso de ingesta.
    Usa el mismo modelo de embeddings para garantizar compatibilidad.
    """
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vectorstore = FAISS.load_local(
        faiss_dir,
        embeddings,
        allow_dangerous_deserialization=True,  # requerido por LangChain para .pkl
    )
    print(f"Índice FAISS cargado desde '{faiss_dir}'")
    return vectorstore

In [6]:
# ──  RECUPERAR CHUNKS RELEVANTES ───────────────────────────────────
def recuperar_chunks(vectorstore: FAISS, query: str, top_k: int = TOP_K) -> list[Document]:
    """
    Ejecuta búsqueda semántica en FAISS.
    Retorna los top_k chunks más relevantes para la consulta.
    """
    chunks = vectorstore.similarity_search(query, k=top_k)
    return chunks

In [7]:
# ── FORMATEAR CONTEXTO CON CITAS ─────────────────────────────────
def formatear_contexto(chunks: list[Document]) -> tuple[str, str]:
    """
    Construye dos cadenas:
    - contexto  : el texto de cada chunk numerado, para el prompt
    - citas_str : las citas formateadas [doc_id | page | chunk_id]

    Retorna (contexto, citas_str)
    """
    bloques_contexto = []
    bloques_citas = []

    for i, chunk in enumerate(chunks, start=1):
        meta = chunk.metadata
        doc_id   = meta.get("doc_id", "desconocido")
        page     = meta.get("page", 0)
        chunk_id = meta.get("chunk_id", f"chunk_{i}")

        # Bloque de contexto numerado
        bloques_contexto.append(
            f"[{i}] {chunk.page_content}"
        )
        # Cita trazable
        bloques_citas.append(
            f"  [{i}] doc_id={doc_id} | page={page} | chunk_id={chunk_id}"
        )

    contexto  = "\n\n".join(bloques_contexto)
    citas_str = "\n".join(bloques_citas)
    return contexto, citas_str

In [8]:
# ── CONSTRUIR PROMPT ──────────────────────────────────────────────
def construir_prompt(query: str, contexto: str) -> str:
    """
    Construye el prompt que obliga al LLM a responder SOLO
    con el contexto recuperado y a referenciar las citas por número [1], [2]…
    """
    prompt = f"""Eres un tutor experto en Biología Celular y Molecular.
Responde la pregunta del estudiante usando ÚNICAMENTE la información
de los fragmentos del corpus proporcionados a continuación.

REGLAS:
1. Si la información no está en los fragmentos, responde: "No encontré esa información en el corpus."
2. Cita las fuentes usando el número del fragmento: [1], [2], etc.
3. Sé claro, preciso y educativo en tu respuesta.

──────────────────────────────────────────
FRAGMENTOS DEL CORPUS:
{contexto}
──────────────────────────────────────────

PREGUNTA: {query}

RESPUESTA:"""
    return prompt

In [9]:
# ── GENERAR RESPUESTA CON GEMINI ─────────────────────────────────
def generar_respuesta(prompt: str) -> str:
    """
    Envía el prompt a Gemini y retorna la respuesta generada.
    """
    llm = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        google_api_key=GOOGLE_API_KEY,
        temperature=0.2,   # baja temperatura para respuestas más fieles al contexto
    )
    respuesta = llm.invoke(prompt)
    return respuesta.content

In [11]:
# ── PIPELINE RAG MÍNIMO ────────────────────────────────
def rag_query(vectorstore: FAISS, query: str) -> dict:
    """
    Pipeline RAG completo de extremo a extremo:
      1. Recupera chunks relevantes desde FAISS
      2. Formatea contexto y citas
      3. Construye prompt
      4. Genera respuesta con Gemini
      5. Retorna resultado con trazabilidad completa

    Retorna un dict con:
      - query        : pregunta original
      - respuesta    : texto generado por Gemini
      - citas        : metadatos de los chunks usados
      - chunks_usados: los chunks recuperados (para trazabilidad)
      - prompt       : el prompt exacto enviado al LLM
    """
    print(f"\n Consulta: {query}")
    print("─" * 60)

    # Paso 1: Recuperar
    chunks = recuperar_chunks(vectorstore, query)
    print(f"Chunks recuperados: {len(chunks)}")

    # Paso 2: Formatear contexto y citas
    contexto, citas_str = formatear_contexto(chunks)

    # Paso 3: Construir prompt
    prompt = construir_prompt(query, contexto)

    # Paso 4: Generar respuesta
    print("Generando respuesta con Gemini...")
    respuesta = generar_respuesta(prompt)

    # Paso 5: Mostrar resultado
    print("\n RESPUESTA:")
    print(respuesta)
    print("\n CITAS TRAZABLES:")
    print(citas_str)

    return {
        "query"        : query,
        "respuesta"    : respuesta,
        "citas"        : citas_str,
        "chunks_usados": chunks,
        "prompt"       : prompt,
    }

In [18]:
if __name__ == "__main__":
    # Cargar índice FAISS
    vectorstore = cargar_vectorstore(FAISS_DIR)

    # Prueba con una pregunta de ejemplo
    resultado = rag_query(
        vectorstore,
        query="¿Que es el endoplasmic reticulum stress (ERS)?"
    )

Índice FAISS cargado desde 'faiss_index'

 Consulta: ¿Que es el endoplasmic reticulum stress (ERS)?
────────────────────────────────────────────────────────────
Chunks recuperados: 4
Generando respuesta con Gemini...

 RESPUESTA:
El endoplasmic reticulum stress (ERS) se refiere a una cascada de señalización intracelular conocida como el sistema de control de calidad que las células activan para prevenirlo [2, 3]. Cuando se generan excesivamente proteínas mal plegadas, asociado a mecanismos de defensa comprometidos, se produce disfunción del orgánulo y muerte celular [2, 3]. El ERS puede activar mecanismos de respuesta celular que se conocen colectivamente como la Respuesta a Proteínas Mal Plegadas (UPR) [2, 3]. Además, el estrés oxidativo inducido por el ERS exacerba la alteración del metabolismo energético en las células de la retina a través de la disfunción mitocondrial [4].

 CITAS TRAZABLES:
  [1] doc_id=Targeting_endoplasmic_reticulum_stress_in_diabetic_retinopathy_mechanistic_in